In [ ]:
from src.compute_text_representations import compute_text_representations
import pandas as pd
import torch
from src.utils import device, encode_df
from src.pairwise_dataset import PairwiseDataset
import numpy as np
from tqdm.auto import tqdm
import xarray as xr
import plotly.express as px
import scipy.stats as stats

In [ ]:
df = pd.read_csv("datasets/relative_clause.csv")
sentences = df["sentence"].tolist()
df = df.drop(columns="sentence")
embeddings = compute_text_representations(
    sentences, model_name="bert-base-uncased", token_aggregation="mean"
)
X = encode_df(df).to(device)
Y = embeddings[5].to(device)
dataset = PairwiseDataset(X, Y, n_pairs=512)

In [ ]:
features = df.columns
x, y = torch.triu_indices(len(features), len(features))
feature_pairs = np.array(
    [f + ", " + f_ for f, f_ in zip(features[x], features[y])]
)

In [ ]:
data = []
for i in tqdm(range(30)):
    X_batch, _ = dataset[0]
    X_batch = (X_batch[:, None] * X_batch[:, :, None])[:, x, y]
    corrs = torch.corrcoef(X_batch.T).cpu()
    corrs = xr.DataArray(
        corrs,
        dims=["feature_pair_1", "feature_pair_2"],
        coords=[feature_pairs, feature_pairs],
    )
    data.append(corrs)
data = xr.concat(data, dim="sample")

In [ ]:
from pingouin import pairwise_corr

In [ ]:
corr, pval = stats.pearsonr(
    X_batch.cpu()[:, :, None], X_batch.cpu()[:, None], axis=0
)

In [ ]:
pcorr = pairwise_corr(pd.DataFrame(X_batch.cpu(), columns=feature_pairs))

In [ ]:
a = np.stack(pcorr["CI95%"].values)

In [ ]:
(a[:, 1] - a[:, 0]).max()

In [ ]:
def correlation_ci_fisher(r, n, alpha=0.05):
    """
    Calculates the confidence interval for a correlation coefficient
    using Fisher's z-transformation. Minimal version without bootstrapping.

    Args:
        r (float): The calculated correlation coefficient (Pearson or Spearman approx).
        n (int): The sample size used to calculate r.
        alpha (float): Significance level (e.g., 0.05 for 95% CI). Defaults to 0.05.

    Returns:
        tuple: (lower_bound, upper_bound) of the confidence interval.
               Returns (np.nan, np.nan) if n <= 3 or invalid r.
    """
    if not (0 < alpha < 1) or not (-1 <= r <= 1):
        return (np.nan, np.nan)

    if n <= 3:
        # Fisher transformation requires n > 3
        return (np.nan, np.nan)

    if abs(r) == 1.0:
        # Handle perfect correlation case
        return (float(r), float(r))

    z = np.arctanh(r)
    se_z = 1.0 / np.sqrt(n - 3.0)
    z_crit = stats.norm.ppf(1.0 - alpha / 2.0)

    z_lower = z - z_crit * se_z
    z_upper = z + z_crit * se_z

    ci_lower = np.tanh(z_lower)
    ci_upper = np.tanh(z_upper)

    return (ci_lower, ci_upper)


# --- Example Usage ---
r_example = 0.65
n_example = 150
alpha_example = 0.05  # 95% CI

ci = correlation_ci_fisher(r_example, n_example, alpha=alpha_example)
print(f"Correlation r: {r_example}")
print(f"Sample size n: {n_example}")
print(f"Alpha: {alpha_example}")
print(f"Confidence Interval (Fisher): ({ci[0]:.4f}, {ci[1]:.4f})")

ci_invalid_n = correlation_ci_fisher(0.5, 5)
print(f"\nCI for n=3: {ci_invalid_n}")

ci_perfect = correlation_ci_fisher(1.0, 100)
print(f"CI for r=1: {ci_perfect}")

In [ ]:
np.sqrt((1 - data**2) / 30)